# 06 Fine Tuning Batch (пакетная тонкая настройка)

Этот notebook (ноутбук) запускает `fine tuning (тонкую настройку)` для `236 strategies (236 стратегий)` из файла:

`D:\Filter_2_option_v2\05_select_for_fine_calibration_output\02d_selected_for_fine_with_rescue.csv`

Рабочий код лежит в `fine_tuning_batch.py (файле Python)`, а результаты сохраняются на диск D:

`D:\Filter_2_option_v2\06_fine_tuning_output`

## Метод

Для trend strategies (трендовых стратегий) используется `local fine grid (локальная тонкая сетка)` вокруг coarse winner (победителя грубой калибровки): 4 ближайшие `T1/T2 (экспирации первого/второго слоя)` считаются отдельно, `K shifts (сдвиги страйка)` и `std spans (окна волатильности)` тоже строятся отдельно для двух слоев. Если coarse winner (победитель грубой калибровки) лежит на краю допустимой сетки, добавляется внешний сосед для проверки продолжения plateau (плато).

Для mean reversion strategies (стратегий возврата к среднему) используется local grid (локальная сетка) вокруг coarse winner (победителя грубой калибровки): `T1 (экспирация первого слоя)` остается узкой, а `T2 (экспирация второго слоя)` расширена через `half-life (период полураспада)` и фиксированные сроки. `K1 (сдвиг первого страйка)` расширен до 4 ближайших значений. Основной `center/scale mode (режим центра и масштаба)` - `median_mad (медиана и медианное абсолютное отклонение)`, а `mean_std (среднее и стандартное отклонение)` подключается только как fallback (резерв), если основной режим слабый.

`BATCH_SIZE (размер пакета)` оставлен `80`. Для `236 strategies (236 стратегий)` нужны `BATCH_ID (номера пакетов)` `0`, `1`, `2`.

In [1]:
from pathlib import Path
import subprocess
import sys

WORK_ROOT = Path(r"./workspace")
SCRIPT = WORK_ROOT / "fine_tuning_batch.py"

cmd = [sys.executable, str(SCRIPT), "--mode", "summary"]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

Selected for calibration (отобрано для калибровки): 236
Batch size (размер пакета): 80
Batch ids (номера пакетов): 0 .. 2
Coarse trend tuples per strategy (грубые трендовые кортежи на стратегию): 216
Coarse mean reversion primary tuples per strategy (грубые основные кортежи возврата к среднему на стратегию): about 5*5*3*3 = 225 before unique half-life clipping
filter_group
put/call     120
call/put      82
call/call     22
put/put       12




## Запуск одного batch (пакета)

Меняй только `BATCH_ID (номер пакета)`. По умолчанию используется `fine mode (тонкий режим)`.

In [ ]:
BATCH_ID = 2
BATCH_SIZE = 80

cmd = [
    sys.executable,
    str(SCRIPT),
    "--mode", "batch",
    "--batch-id", str(BATCH_ID),
    "--batch-size", str(BATCH_SIZE),
    "--calibration-mode", "fine",
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

## Audit (аудит) и combine (объединение)

In [5]:
cmd = [sys.executable, str(SCRIPT), "--mode", "audit"]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

{
  "selected_rows": 236,
  "processed_rows": 236,
  "processed_unique": 236,
  "missing_count": 0,
  "duplicate_count": 0,
  "missing_spec_id_ranges": [],
  "duplicate_head": []
}




In [6]:
cmd = [sys.executable, str(SCRIPT), "--mode", "combine"]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

{
  "n_best_batch_files": 3,
  "n_unique_strategies": 236,
  "n_best_paths": 236,
  "output_dir": "D:\\Filter_2_option_v2\\06_fine_tuning_output"
}




## Результаты

Главные files (файлы):

- `batches/grid_batch_XXXX.csv` - полная fine grid (тонкая сетка) одного batch (пакета);
- `batches/best_batch_XXXX.csv` - лучший вариант на стратегию;
- `03_calibrated_best_by_strategy.csv` - итоговые лучшие параметры;
- `04_calibrated_best_paths.csv` - лучшие trajectories (траектории);
- `05_manifest.json` - manifest (паспорт сборки).